# Zomato Delhi — Data Cleaning + EDA (Case study: Khuyến mãi)

Notebook làm sạch dữ liệu và phân tích khám phá (EDA) cho dataset Zomato Delhi — case study bổ sung, tập trung vào tác động của khuyến mãi lên doanh thu.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
pd.set_option('display.max_columns', 50)

## 0. ĐƯỜNG DẪN (tự động tìm đúng thư mục gốc project, chạy từ đâu cũng được)

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent  # khi chạy dạng .py
except NameError:
    BASE_DIR = Path.cwd().resolve().parent  # khi chạy trong Jupyter Notebook (đứng trong notebooks/)
RAW_PATH = BASE_DIR / 'data' / 'raw' / 'order_history_kaggle_data.csv'
CLEAN_DIR = BASE_DIR / 'data' / 'clean'
CHARTS_DIR = BASE_DIR / 'outputs' / 'charts'
KPI_DIR = BASE_DIR / 'outputs' / 'kpi_summary'
for d in [CLEAN_DIR, CHARTS_DIR, KPI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. LOAD DATA

In [3]:
df = pd.read_csv(RAW_PATH)
print("Shape ban đầu:", df.shape)

Shape ban đầu: (21321, 29)


## 2. LOẠI CỘT THIẾU QUÁ NHIỀU (>95%) VÀ CỘT KHÔNG CÓ BIẾN THIÊN

In [4]:
cols_to_drop = [
    'Instructions', 'Review', 'Cancellation/Rejection reason',
    'Restaurant compensation (Cancellation)', 'Restaurant penalty (Rejection)',
    'Customer complaint tag', 'City', 'Delivery'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Đã loại {len(cols_to_drop)} cột thiếu quá nhiều / không biến thiên")

Đã loại 8 cột thiếu quá nhiều / không biến thiên


## 3. PARSE THỜI GIAN

In [5]:
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y', errors='coerce')
print(f"Số dòng không parse được thời gian: {df['Order Placed At'].isna().sum()}")
df['order_hour'] = df['Order Placed At'].dt.hour
df['order_dayofweek'] = df['Order Placed At'].dt.day_name()

# ------------------------------------------------------------
# 4. CHUẨN HÓA CỘT DISTANCE (text -> số km)
#    Quy ước theo xác nhận của user: "<1km" -> 1km
# ------------------------------------------------------------
def parse_distance(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower().replace('km', '')
    if val.startswith('<'):
        return 1.0
    try:
        return float(val)
    except ValueError:
        return np.nan

df['distance_km'] = df['Distance'].apply(parse_distance)
print(f"Số dòng không parse được Distance: {df['distance_km'].isna().sum()}")

Số dòng không parse được thời gian: 0
Số dòng không parse được Distance: 0


## 5. XỬ LÝ CỘT DISCOUNT CONSTRUCT (missing = không áp dụng khuyến mãi)

In [6]:
df['Discount construct'] = df['Discount construct'].fillna('No Discount')

## 6. TẠO CỘT TỔNG KHUYẾN MÃI

In [7]:
discount_cols = [
    'Restaurant discount (Promo)',
    'Restaurant discount (Flat offs, Freebies & others)',
    'Gold discount',
    'Brand pack discount'
]
df['total_discount'] = df[discount_cols].sum(axis=1)
df['has_discount'] = df['total_discount'] > 0

## 7. XỬ LÝ RATING (thiếu 88.3%) - KHÔNG điền giá trị giả

In [8]:
print(f"\nSố đơn có Rating: {df['Rating'].notna().sum()} / {len(df)} "
      f"({df['Rating'].notna().mean()*100:.1f}%)")
df_rating = df[df['Rating'].notna()].copy()  # dùng riêng khi phân tích Rating


Số đơn có Rating: 2491 / 21321 (11.7%)


## 8. KIỂM TRA OUTLIER

In [9]:
print("\nThống kê mô tả các cột số chính:")
print(df[['Bill subtotal', 'Total', 'distance_km', 'KPT duration (minutes)',
           'Rider wait time (minutes)']].describe())


Thống kê mô tả các cột số chính:
       Bill subtotal         Total   distance_km  KPT duration (minutes)  \
count   21321.000000  21321.000000  21321.000000            21026.000000   
mean      750.076838    682.616113      4.182731               17.332960   
std       498.759428    465.313977      2.977212                6.283388   
min        50.000000     52.500000      1.000000                0.000000   
25%       459.000000    387.450000      2.000000               13.380000   
50%       629.000000    597.450000      3.000000               16.330000   
75%       899.000000    837.900000      6.000000               20.050000   
max     16080.000000  12663.000000     21.000000               90.870000   

       Rider wait time (minutes)  
count               21153.000000  
mean                    4.825070  
std                     4.982591  
min                     0.100000  
25%                     1.000000  
50%                     3.100000  
75%                     7.400000  
m

## 9. LƯU FILE ĐÃ LÀM SẠCH

In [10]:
df.to_csv(CLEAN_DIR / 'zomato_clean.csv', index=False)
print(f"\nĐã lưu zomato_clean.csv | Shape cuối: {df.shape}")

# ============================================================
# EDA - PHÂN TÍCH KHÁM PHÁ DỮ LIỆU
# ============================================================

# Chỉ tính Revenue/AOV trên đơn thành công (Delivered) - theo xác nhận của user
df_delivered = df[df['Order Status'] == 'Delivered'].copy()
print(f"\nSố đơn 'Delivered' dùng để tính Revenue/AOV: {len(df_delivered)} / {len(df)}")

# --- Q7: Khuyến mãi có làm tăng số lượng đơn hàng không? ---
orders_by_discount = df_delivered.groupby('has_discount').size()
print("\n=== Số lượng đơn theo có/không khuyến mãi (đơn Delivered) ===")
print(orders_by_discount)

# --- Q8: Khuyến mãi có làm giảm AOV không? ---
aov_by_discount = df_delivered.groupby('has_discount')['Total'].mean()
print("\n=== AOV theo có/không khuyến mãi (đơn Delivered) ===")
print(aov_by_discount)

corr_discount_bill = df_delivered['total_discount'].corr(df_delivered['Bill subtotal'])
print(f"\nTương quan Tổng khuyến mãi vs Bill subtotal: {corr_discount_bill:.3f}")

plt.figure(figsize=(7, 5))
sns.barplot(x=aov_by_discount.index.map({True: 'Có khuyến mãi', False: 'Không khuyến mãi'}),
            y=aov_by_discount.values)
plt.title('AOV: Có khuyến mãi vs Không khuyến mãi')
plt.ylabel('AOV (₹)')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_aov_by_discount.png', dpi=120)
plt.close()

plt.figure(figsize=(7, 5))
sns.barplot(x=orders_by_discount.index.map({True: 'Có khuyến mãi', False: 'Không khuyến mãi'}),
            y=orders_by_discount.values)
plt.title('Số lượng đơn: Có khuyến mãi vs Không khuyến mãi')
plt.ylabel('Số đơn')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_orders_by_discount.png', dpi=120)
plt.close()

# --- Q9: Đơn hủy/từ chối có liên quan đến KPT duration không? ---
status_kpt = df.groupby('Order Status')['KPT duration (minutes)'].mean().sort_values(ascending=False)
print("\n=== Thời gian chuẩn bị (KPT) trung bình theo Trạng thái đơn ===")
print(status_kpt)

plt.figure(figsize=(9, 5))
status_kpt.plot(kind='bar', color='indianred')
plt.title('Thời gian chuẩn bị món (KPT) trung bình theo Trạng thái đơn')
plt.ylabel('KPT (phút)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'chart_kpt_by_status.png', dpi=120)
plt.close()

order_status_dist = df['Order Status'].value_counts(normalize=True) * 100
print("\n=== Phân bố Order Status (%) ===")
print(order_status_dist)

# --- Tính KPI tổng hợp để so sánh liên thị trường ---
kpi_summary = {
    'dataset': 'Zomato',
    'market': 'India (Delhi NCR)',
    'total_orders_delivered': len(df_delivered),
    'total_revenue': df_delivered['Total'].sum(),
    'aov': df_delivered['Total'].mean(),
    'pct_orders_with_discount': df_delivered['has_discount'].mean() * 100,
    'aov_with_discount': aov_by_discount.get(True, np.nan),
    'aov_without_discount': aov_by_discount.get(False, np.nan),
    'corr_discount_vs_bill_subtotal': corr_discount_bill,
    'pct_rating_available': df['Rating'].notna().mean() * 100,
}
pd.DataFrame([kpi_summary]).to_csv(KPI_DIR / 'zomato_kpi_summary.csv', index=False)

print("\n=== HOÀN TẤT CLEANING + EDA ZOMATO ===")
print("Các file đã tạo: zomato_clean.csv, zomato_kpi_summary.csv, và 3 biểu đồ PNG")


Đã lưu zomato_clean.csv | Shape cuối: (21321, 27)

Số đơn 'Delivered' dùng để tính Revenue/AOV: 21131 / 21321

=== Số lượng đơn theo có/không khuyến mãi (đơn Delivered) ===
has_discount
False     8217
True     12914
dtype: int64

=== AOV theo có/không khuyến mãi (đơn Delivered) ===
has_discount
False    710.017024
True     665.105302
Name: Total, dtype: float64

Tương quan Tổng khuyến mãi vs Bill subtotal: 0.504



=== Thời gian chuẩn bị (KPT) trung bình theo Trạng thái đơn ===
Order Status
Picked up           19.590000
Delivered           17.339428
Returned            16.673600
Rejected            15.516721
Return cancelled    12.366667
Timed out                 NaN
Name: KPT duration (minutes), dtype: float64

=== Phân bố Order Status (%) ===
Order Status
Delivered           99.108860
Rejected             0.741053
Returned             0.117255
Return cancelled     0.014071
Picked up            0.014071
Timed out            0.004690
Name: proportion, dtype: float64

=== HOÀN TẤT CLEANING + EDA ZOMATO ===
Các file đã tạo: zomato_clean.csv, zomato_kpi_summary.csv, và 3 biểu đồ PNG
